In [15]:
import pandas as pd
import glob
import os

file = r"E:\Om\mllm\data\video_metadata\@walgreens.json"
import json

with open(file, "r", encoding="utf-8") as f:
    data = json.load(f)

# Convert JSON videos to DataFrame
df = pd.DataFrame(data["videos"])
df["handle"] = data.get("seed", "")


In [31]:
df[df["video_duration"]<70]

,id,title,description,publishedAt,channelId,channelTitle,tags,categoryId,defaultLanguage,duration,definition,caption,viewCount,likeCount,commentCount,video_duration,handle,embedding,cluster
0,grb86txLIf8,A Moment to Remember,It’s not just a card. It’s a keepsake you’ll a...,2025-12-12T23:36:36Z,UCsVIERusF2kZEHUB1x1lhZA,Walgreens,[],22,en,PT1M5S,hd,false,254,6,0,65,@walgreens,"[0.01393346767872572, -0.006156278774142265, -...",-1
3,AceFaYhicpo,Holiday Road | Walgreens,"The holiday road may be long, but through ever...",2025-11-17T16:09:57Z,UCsVIERusF2kZEHUB1x1lhZA,Walgreens,[],22,en,PT16S,hd,false,1665,20,7,16,@walgreens,"[0.01682155393064022, 0.01290239579975605, -0....",1
4,WsEvS0rfjMk,Holiday Road | Walgreens,"The holiday road may be long, but through ever...",2025-11-17T16:09:46Z,UCsVIERusF2kZEHUB1x1lhZA,Walgreens,[],22,en,PT31S,hd,false,5159,56,35,31,@walgreens,"[0.01682155393064022, 0.01290239579975605, -0....",1
16,SBv-_PEiuFA,Reframe menopause with Walgreens #MenopauseIsHot,The stigma around menopause has kept women fro...,2025-07-11T22:53:57Z,UCsVIERusF2kZEHUB1x1lhZA,Walgreens,[],22,en,PT31S,hd,false,2975,13,14,31,@walgreens,"[0.03207537904381752, 0.0387442484498024, -0.0...",2
32,TP3kqC5Gin8,Happy Halloween! 🎃👻 Which costume did you pick...,,2024-10-31T15:41:02Z,UCsVIERusF2kZEHUB1x1lhZA,Walgreens,[],22,en,PT16S,hd,false,4134,121,4,16,@walgreens,"[0.023391230031847954, -0.012540780939161777, ...",-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
389,hsQ4RV5ieVM,Do I need hand sanitizer to prevent COVID-19 (...,Do I need hand sanitizer to prevent COVOID-19 ...,2020-03-12T16:30:52Z,UCsVIERusF2kZEHUB1x1lhZA,Walgreens,"[coronavirus, health, wellness, walgreens, cov...",27,en,PT26S,hd,false,11358,38,3,26,@walgreens,"[0.03197256848216057, -0.02420813776552677, -0...",-1
390,hbR5gcAcLxc,What does it mean to be quarantined from COVID...,What does it mean to be quarantined from COVID...,2020-03-11T18:53:06Z,UCsVIERusF2kZEHUB1x1lhZA,Walgreens,"[coronavirus, covid-19, health, flu, wellness,...",25,en,PT40S,hd,false,9393,40,4,40,@walgreens,"[-0.023406554013490677, -0.037042371928691864,...",-1
391,1HZJiMqFB3I,Walgreens | TopKnot,Take your top knot to new levels with our pro ...,2018-08-02T19:24:17Z,UCsVIERusF2kZEHUB1x1lhZA,Walgreens,"[walgreens, hair care, hair, hair styles, hair...",27,en,PT26S,hd,true,2790,45,0,26,@walgreens,"[0.004179257899522781, 0.03021053783595562, -0...",-1
392,dqdQGaFCV-4,Print Photos from Instagram | Walgreens iPhone...,Print your Instagram photos right from your ph...,2015-08-27T19:47:33Z,UCsVIERusF2kZEHUB1x1lhZA,Walgreens,"[print from instagram, printing photos, photo ...",27,en,PT16S,hd,false,4212,30,3,16,@walgreens,"[-0.009725091978907585, 0.020023183897137642, ...",59


In [17]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# 1) Load variables from .env into environment
load_dotenv()

# 2) Read the key from env
api_key = os.getenv("OPENAI_API_KEY")
if api_key is None:
    raise RuntimeError("OPENAI_API_KEY not set in .env")

# 3) Create client
client = OpenAI(api_key=api_key)

def row_to_text(row):
    return (
        f"Title: {row['title']}\n"
        f"Channel: {row['channelTitle']} ({row['handle']})\n"
        # f"Duration: {row['video_duration']} seconds\n"
        f"Description: {row['description']}"
    )

texts = df.apply(row_to_text, axis=1).tolist()

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=texts,
)

embeddings = [d.embedding for d in response.data]
df["embedding"] = embeddings


In [18]:
import numpy as np

X = np.vstack(df["embedding"].values)   # shape: (n_videos, 1536)


In [19]:
from sklearn.cluster import DBSCAN

# Tune eps / min_samples as needed
dbscan = DBSCAN(
    eps=0.2,              # try 0.1–0.4 range
    min_samples=2,
    metric="cosine",
    n_jobs=-1             # use all cores
)

labels = dbscan.fit_predict(X)

df["cluster"] = labels


In [20]:
print(df["cluster"].value_counts())


cluster
-1     186
 17     34
 53      8
 44      8
 22      7
      ... 
 55      2
 56      2
 57      2
 58      2
 59      2
Name: count, Length: 61, dtype: int64


In [21]:
for c in sorted(df["cluster"].unique()):
    print(f"\n=== Cluster {c} ===")
    sample = df[df["cluster"] == c].head(5)
    for _, row in sample.iterrows():
        print(f"- {row['title']}  |  {row['channelTitle']}")



=== Cluster -1 ===
- A Moment to Remember  |  Walgreens
- How One Walgreens Team Member Saved a Life | Act of Heroism  |  Walgreens
- 🎓 Twin Pharmacists on a Mission: Meet Sean & Stephen Bly | #WeAreWalgreens  |  Walgreens
- Walgreens Vaccine B roll  |  Walgreens
- From Pharmacy Tech to Future Pharmacist | Tyrone’s Walgreens Journey  |  Walgreens

=== Cluster 0 ===
- How Walgreens Is Opening Clinical Trials to More Patients | Patrick Gee’s Story  |  Walgreens
- Walgreens Clinical Trials | Patient Story  |  Walgreens
- Walgreens Clinical Trials Event 2023  |  Walgreens
- Building the future of clinical trials with Walgreens  |  Walgreens

=== Cluster 1 ===
- Holiday Road | Walgreens  |  Walgreens
- Holiday Road | Walgreens  |  Walgreens

=== Cluster 2 ===
- Reframe menopause with Walgreens #MenopauseIsHot  |  Walgreens
- Menopause is Hot - Short Documentary  |  Walgreens

=== Cluster 3 ===
- Walgreens History: The First Store Outside of Illinois | Visualizing Our Past  |  Walgreens
- W

In [22]:
titles_with_silver = df[df['title'].str.lower().str.contains("silver", na=False)]['title']
print(titles_with_silver)

Series([], Name: title, dtype: object)


In [25]:
def row_to_text(row):
    return (
        f"Title: {row['title']}\n"
        f"Channel: {row['channelTitle']} ({row['handle']})\n"
        f"Description: {row['description']}"
    )

texts = df.apply(row_to_text, axis=1).tolist()

In [26]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

def match_pairs_within_cluster_relative(
    dfc: pd.DataFrame,
    ratio_min: float = 1.5,
    ratio_max: float = 6.0,
    min_cos: float = 0.75,
    long_capacity: int = 3,
    top_k_per_short: int = 5
):
    dfc = dfc.reset_index(drop=True).copy()
    n = len(dfc)
    if n < 2:
        return pd.DataFrame(columns=[
            "cluster",
            "short_id", "long_id",
            "cos_sim",
            "short_dur", "long_dur",
            "short_title", "long_title",
            "ratio",
        ])

    X = np.vstack(dfc["embedding"].values)
    cos = cosine_similarity(X, X)

    dur = dfc["video_duration"].to_numpy().astype(float)
    di = dur[:, None]
    dj = dur[None, :]

    ratio = dj / np.maximum(di, 1.0)
    valid = (dj > di) & (ratio >= ratio_min) & (ratio <= ratio_max)
    np.fill_diagonal(valid, False)

    score = np.where(valid, cos, -1e9)

    edges = []
    for i in range(n):
        js = np.argsort(-score[i])[:top_k_per_short]
        for j in js:
            if score[i, j] < min_cos:
                continue
            if score[i, j] <= -1e8:
                continue
            edges.append((i, j, float(score[i, j])))

    edges.sort(key=lambda x: x[2], reverse=True)
    cap = {j: long_capacity for j in range(n)}
    used_short = set()
    pairs = []

    for i, j, s in edges:
        if i in used_short:
            continue
        if cap[j] <= 0:
            continue
        used_short.add(i)
        cap[j] -= 1

        pairs.append({
            "cluster": dfc["cluster"].iloc[0] if "cluster" in dfc.columns else None,
            "short_id": dfc.loc[i, "id"],
            "long_id":  dfc.loc[j, "id"],
            "cos_sim":  s,
            "short_dur": int(dfc.loc[i, "video_duration"]),
            "long_dur":  int(dfc.loc[j, "video_duration"]),
            "short_title": dfc.loc[i, "title"],
            "long_title":  dfc.loc[j, "title"],
            "ratio": float(dfc.loc[j, "video_duration"] / max(dfc.loc[i, "video_duration"], 1)),
        })

    if not pairs:
        return pd.DataFrame(columns=[
            "cluster",
            "short_id", "long_id",
            "cos_sim",
            "short_dur", "long_dur",
            "short_title", "long_title",
            "ratio",
        ])

    out = pd.DataFrame(pairs)
    return out.sort_values("cos_sim", ascending=False).reset_index(drop=True)



def match_all_clusters_relative(df: pd.DataFrame, **kwargs):
    outs = []
    for c, dfc in df.groupby("cluster"):
        outs.append(match_pairs_within_cluster_relative(dfc, **kwargs))
    return pd.concat(outs, ignore_index=True) if outs else pd.DataFrame()


In [27]:
pairs = match_all_clusters_relative(
    df,
    ratio_min=1.5,
    ratio_max=6.0,
    min_cos=0.75,
    long_capacity=3,
    top_k_per_short=5
)

C:\Users\omras\AppData\Local\Temp\ipykernel_13716\1681140510.py:92: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(outs, ignore_index=True) if outs else pd.DataFrame()


In [18]:
pairs.to_csv("matched_pairs.csv", index=False)

In [28]:
pairs

,cluster,short_id,long_id,cos_sim,short_dur,long_dur,short_title,long_title,ratio
0,-1,TwAjIMuoi6M,vtpTCiHfn_I,0.795877,30,76,Mito: Las personas que ya han contraído COVID-...,Mito: COVID-19 no representa un peligro para l...,2.533333
1,-1,QpV9rasGq4w,u16u-Rtj2fA,0.792917,123,196,How to Use An Eyebrow Pencil with Jen Brown | ...,How to groom your eyebrows with Jen Brown | Wa...,1.593496
2,-1,32c73q8GqCY,u16u-Rtj2fA,0.788142,80,196,How to Create Fluffy Glued Brows with Jen Pham...,How to groom your eyebrows with Jen Brown | Wa...,2.450000
3,-1,zO7VDr8F0o8,5GoqryQrLYo,0.783082,31,79,Myth: People that have already gotten COVID-19...,"If I already had COVID-19, do I still need to ...",2.548387
4,-1,new8dgIYuiw,Um47zzhK294,0.781439,60,187,Supporting Black Women’s Wellness with Flu & C...,COVID-19 Vaccine Equity Initiative To Expand A...,3.116667
...,...,...,...,...,...,...,...,...,...
70,53,3Igbp1YSEmM,KJ-UDzYxCzM,0.837706,7,32,All of Us Programa Científico - El Poder,All of Us Programa Científico,4.571429
71,53,s6AdsfJct6E,eh1Bcu-bM9g,0.766743,16,51,All of Us Research Program - Kolbi Testimonial...,All of Us Research Program,3.187500
72,53,eh1Bcu-bM9g,YIwxaerkGec,0.766735,51,142,All of Us Research Program,All of Us Research Program - Kolbi Testimonial...,2.784314
73,53,KJ-UDzYxCzM,eh1Bcu-bM9g,0.759369,32,51,All of Us Programa Científico,All of Us Research Program,1.593750


In [29]:
unique_video_ids = set(pairs["short_id"]).union(pairs["long_id"])
print(f"Number of unique video ids in pairs: {len(unique_video_ids)}")


Number of unique video ids in pairs: 125


In [32]:
import json
import glob
import os

# Paths (update paths if needed)
channel_logs_path = r"E:\Om\mllm\data\logs\channel_logs.json"
video_metadata_glob = r"E:\Om\mllm\data\video_metadata\*.dnwd.json"

# Load channel logs JSON
with open(channel_logs_path, "r", encoding="utf-8") as f:
    channel_logs = json.load(f)

# Build handle to video count mapping
handle_to_video_count = {}
for vm_path in glob.glob(video_metadata_glob):
    base = os.path.basename(vm_path)
    # file is like '@4imprint.dnwd.json' -> '@4imprint'
    if base.endswith(".dnwd.json"):
        handle = base[:-len(".dnwd.json")]
    else:
        handle = base.split('.')[0]
    with open(vm_path, "r", encoding="utf-8") as f:
        try:
            data = json.load(f)
            # Each file is assumed to be a list of video dicts
            count = len(data)
        except Exception:
            count = 0
    handle_to_video_count[handle] = count

# Update "Initial Screening" for matching handle
for entry in channel_logs:
    handle = entry.get("handle")
    if handle in handle_to_video_count:
        entry["stage"]["Initial Screening"] = handle_to_video_count[handle]

# Save updated channel logs
with open(channel_logs_path, "w", encoding="utf-8") as f:
    json.dump(channel_logs, f, indent=2, ensure_ascii=False)

print("Updated 'Initial Screening' in channel_logs.json according to DNWD video data.")


Updated 'Initial Screening' in channel_logs.json according to DNWD video data.
